# Fine-tune model FITS (star vs galaxy) dengan dataset baru

Notebook ini melatih ulang (fine-tune) model `fits_star_galaxy_model.keras` dari repo
**astro-classifier** kamu, memakai dataset baru: **[Star-Galaxy Classification Data](https://www.kaggle.com/datasets/divyansh22/dummy-astronomy-data)**
(citra 64x64 dari teleskop 1.3m ARIES, label star/galaxy dari SDSS).

Cukup jalankan sel dari atas ke bawah (Runtime > Run all), atau satu-satu dengan Shift+Enter.

**Sebelum mulai:** aktifkan GPU biar lebih cepat -> menu `Runtime` > `Change runtime type` > `Hardware accelerator` > `T4 GPU` > Save.

## Langkah 0 — Cek GPU (opsional, tapi disarankan)

In [ ]:
!nvidia-smi -L || print('Tidak ada GPU terpasang — training tetap jalan, cuma lebih lambat.')

## Langkah 1 — Ambil source code project kamu
Kita clone repo GitHub kamu supaya bisa pakai `fits_utils.py` (preprocessing) dan model lama sebagai titik awal fine-tuning.

In [ ]:
!git clone https://github.com/naymiwa/astro-classifier.git

%cd astro-classifier

## Langkah 2 — Install dependency yang belum ada di Colab (astropy)

In [ ]:
!pip install -q astropy

## Langkah 3 — Upload dataset baru dari Kaggle

1. Buka **https://www.kaggle.com/datasets/divyansh22/dummy-astronomy-data** (login Kaggle dulu kalau belum).
2. Klik tombol **Download** di kanan atas halaman -> kamu dapat file `archive.zip` (~8.5 MB) di komputer kamu.
3. Jalankan sel di bawah ini, lalu klik **Choose Files** dan pilih `archive.zip` yang baru diunduh.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pilih archive.zip

zip_name = list(uploaded.keys())[0]

print('File terupload:', zip_name)

In [ ]:
import zipfile, os



DATA_DIR = 'real_star_galaxy'

os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name) as z:

    z.extractall(DATA_DIR)



# Tampilkan struktur folder biar kelihatan isinya

for root, dirs, fnames in os.walk(DATA_DIR):

    depth = root.replace(DATA_DIR, '').count(os.sep)

    print('  ' * depth + os.path.basename(root) + '/', f'({len(fnames)} file)' if fnames else '')

## Langkah 4 — Kumpulkan file gambar + label

Script ini otomatis mencari semua file `.jpg`/`.jpeg`/`.png` di dalam `real_star_galaxy/`, lalu
memberi label berdasarkan nama folder (folder yang mengandung kata **star** = label 0,
folder yang mengandung kata **galaxy** = label 1). Ini cocok dengan urutan label di `train_fits.py`.

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png')



samples = []  # (filepath, label)

for root, _, fnames in os.walk(DATA_DIR):

    root_lower = root.lower()

    if 'star' in root_lower:

        label = 0

    elif 'galax' in root_lower:

        label = 1

    else:

        continue

    for fn in fnames:

        if fn.lower().endswith(IMG_EXT):

            samples.append((os.path.join(root, fn), label))



n_star = sum(1 for _, l in samples if l == 0)

n_gal = sum(1 for _, l in samples if l == 1)

print(f'Ditemukan {len(samples)} gambar  ->  star: {n_star}, galaxy: {n_gal}')

assert len(samples) > 0, 'Tidak ada gambar ditemukan — cek struktur folder di atas dan sesuaikan aturan label.'

## Langkah 5 — Preprocessing (sama persis dengan yang dipakai aplikasi)

Kita pakai `fits_utils.to_model_input()` dari repo kamu (normalisasi percentile + resize 64x64 + 3 channel),
supaya cara model 'melihat' data saat training sama persis dengan saat prediksi di aplikasi nanti.

In [ ]:
import numpy as np

from PIL import Image

import fits_utils



def load_and_preprocess(path):

    img = Image.open(path).convert('L')  # grayscale

    arr = np.asarray(img, dtype=np.float32)

    return fits_utils.to_model_input(arr)



X, y = [], []

for path, label in samples:

    try:

        X.append(load_and_preprocess(path))

        y.append(label)

    except Exception as e:

        print('Lewati', path, '->', e)



X = np.stack(X).astype(np.float32)

y = np.asarray(y, dtype=np.int32)

print('X shape:', X.shape, ' y shape:', y.shape)

## Langkah 6 — Split train/validation
Pakai fungsi `stratified_split` yang sama dengan `train_fits.py`, biar proporsi star/galaxy seimbang di kedua sisi.

In [ ]:
import train_fits



train_idx, val_idx = train_fits.stratified_split(y, val_frac=0.2, seed=0)

X_train, y_train = X[train_idx], y[train_idx]

X_val, y_val = X[val_idx], y[val_idx]

print(f'Train: {len(train_idx)}  Val: {len(val_idx)}')

## Langkah 7 — Fine-tune model yang sudah ada

Kita **lanjutkan training** dari `fits_star_galaxy_model.keras` yang sudah ada di repo (bukan mulai dari nol),
dengan learning rate kecil supaya model beradaptasi ke data foto asli tanpa 'lupa' semua yang sudah dipelajari.

In [ ]:
from tensorflow import keras



model = keras.models.load_model('fits_star_galaxy_model.keras')

model.compile(

    optimizer=keras.optimizers.Adam(1e-4),

    loss='binary_crossentropy',

    metrics=['accuracy'],

)



callbacks = [

    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),

    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),

]



history = model.fit(

    X_train, y_train,

    validation_data=(X_val, y_val),

    epochs=30,

    batch_size=32,

    callbacks=callbacks,

    verbose=1,

)

## Langkah 8 — Evaluasi hasil fine-tuning

In [ ]:
probs = model.predict(X_val, verbose=0)[:, 0]

preds = (probs >= 0.5).astype(int)

acc = float((preds == y_val).mean())

print(f'Val accuracy: {acc:.4f}')

for cls, name in [(0, 'star'), (1, 'galaxy')]:

    mask = y_val == cls

    if mask.sum() == 0:

        continue

    acc_cls = float((preds[mask] == y_val[mask]).mean())

    print(f'  {name:7s}: {acc_cls:.4f}  (n={int(mask.sum())})')

## Langkah 9 — Simpan & unduh model baru

Nama file disamakan dengan yang dipakai aplikasi (`fits_star_galaxy_model.keras`), jadi kamu tinggal
menimpa file lama di project lokal kamu.

In [ ]:
model.save('fits_star_galaxy_model.keras')



from google.colab import files

files.download('fits_star_galaxy_model.keras')

## Langkah 10 — Pasang kembali ke project

1. File `fits_star_galaxy_model.keras` otomatis ke-download ke folder Downloads komputer kamu.
2. Salin file itu ke folder project lokal kamu, **timpa** file lama dengan nama yang sama.
3. Jalankan aplikasi seperti biasa (`uvicorn main:app --reload`) lalu tes upload beberapa file FITS.
4. Kalau hasilnya sudah bagus, commit & push:

```powershell
git add fits_star_galaxy_model.keras
git commit -m "Fine-tune model FITS dengan dataset real star/galaxy"
git push
```

> Kalau mau, kamu juga bisa kirim file `.keras` hasil fine-tuning ini ke chat, nanti aku bantu pasang, tes, dan commit di sandbox.